# Demo Chương 5: Phân tích Rò rỉ Mạng qua Mô hình Học máy

**Môn học:** An toàn và Bảo mật Hệ thống Thông tin
**Trọng tâm Demo:** Xây dựng mô hình phân tích lưu lượng mạng để phát hiện Rò rỉ Dữ liệu (Data Leakage)

Trong bài thực hành này, chúng ta tập trung phân tích các luồng kết nối (network flow) nhằm phát hiện kịch bản nội gián (insider threat) - một nhân viên cố tình truyền dữ liệu ra ngoài mạng vượt thẩm quyền ngoài giờ làm việc.

## 1. Sinh dữ liệu giả lập (Data Generation)

Tạo 5,000 bản ghi dữ liệu mạng dạng flow với tỉ lệ 85% bình thường (Normal) và 15% rò rỉ dữ liệu (Leakage).

In [1]:
import sys; sys.path.append('src')
from generate_data import generate_dataset

df = generate_dataset()
df.head()

[*] Sinh 4250 bản ghi Normal và 750 bản ghi Leakage...


[✓] Đã lưu dataset tại: /home/adc300/src/ptit/atbmtt/data/network_logs.csv
    Tổng bản ghi : 5000
    Normal (0)   : 4250 (85.0%)
    Leakage (1)  : 750 (15.0%)

[*] 5 bản ghi đầu tiên:
          timestamp        src_ip        dst_ip  dst_port protocol  bytes_sent  bytes_recv  duration  hour_of_day  is_external_dst  label
2025-10-02 10:15:19  192.168.2.16   45.33.32.23        25      UDP       31.47      144.13      5.87           10                1      0
2025-10-27 14:50:19     10.0.1.64 192.168.1.237      5432      TCP        8.81        5.07     50.97           14                0      0
2025-10-28 16:56:07    10.0.1.250    10.0.2.102       110      TCP       15.62       57.22     12.26           16                0      0
2025-10-28 06:15:53 192.168.1.140 192.168.2.247      3306      UDP       17.40        4.69      1.62            6                0      0
2025-10-13 16:31:12 192.168.1.226 192.168.1.120        80      UDP      135.40      156.57     35.46           16          

,timestamp,src_ip,dst_ip,dst_port,protocol,bytes_sent,bytes_recv,duration,hour_of_day,is_external_dst,label
0,2025-10-02 10:15:19,192.168.2.16,45.33.32.23,25,UDP,31.47,144.13,5.87,10,1,0
1,2025-10-27 14:50:19,10.0.1.64,192.168.1.237,5432,TCP,8.81,5.07,50.97,14,0,0
2,2025-10-28 16:56:07,10.0.1.250,10.0.2.102,110,TCP,15.62,57.22,12.26,16,0,0
3,2025-10-28 06:15:53,192.168.1.140,192.168.2.247,3306,UDP,17.40,4.69,1.62,6,0,0
4,2025-10-13 16:31:12,192.168.1.226,192.168.1.120,80,UDP,135.40,156.57,35.46,16,0,0


## 2. Tiền xử lý dữ liệu (Preprocessing)

Quy trình tiền xử lý bao gồm:
- Làm sạch dữ liệu và loại bỏ các cột không sử dụng (IP, Timestamp).
- Mã hóa các biến phân loại (`protocol`).
- Thay đổi đặc trưng dẫn xuất: `upload_download_ratio = bytes_sent / bytes_recv`.
- Chuẩn hóa các biến liên tục MinMaxScaler (khoảng [0,1]).
- Chia tập huấn luyện/kiểm tra 80:20.

In [2]:
from preprocess import run_preprocessing_pipeline

X_train, X_test, y_train, y_test, scaler, le = run_preprocessing_pipeline()
print('Training set size:', X_train.shape)
print('Testing set size:', X_test.shape)
X_train.head()

  QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU
[*] Đọc dữ liệu: 5000 bản ghi, 11 cột

── Bước 1: Làm sạch dữ liệu ──
  [✓] Không có giá trị thiếu
  [✓] Loại bỏ 0 bản ghi trùng lặp → còn 5000 bản ghi
  [✓] Loại bỏ các cột: ['timestamp', 'src_ip', 'dst_ip']

── Bước 2: Mã hóa biến phân loại ──
  [✓] Label Encoding protocol: {'ICMP': np.int64(0), 'TCP': np.int64(1), 'UDP': np.int64(2)}

── Bước 3: Feature Engineering ──
  [✓] Tạo upload_download_ratio = bytes_sent / bytes_recv
      Min: 0.0002, Max: 141407.5000, Mean: 91.5695

── Chia dữ liệu (80/20 Stratified) ──
  [✓] Train: 4000 bản ghi (Normal: 3400, Leakage: 600)
  [✓] Test : 1000 bản ghi (Normal: 850, Leakage: 150)

── Bước 4: Chuẩn hóa dữ liệu số (MinMaxScaler) ──
  [✓] Chuẩn hóa cột: ['bytes_sent', 'bytes_recv', 'duration']
      (fit trên train → transform trên cả train & test để tránh data leakage)

  [✓] TIỀN XỬ LÝ HOÀN TẤT
  Đặc trưng sử dụng (8): ['bytes_sent', 'bytes_recv', 'duration', 'hour_of_day', 'is_external_dst', 'dst_port', 'protoco

,bytes_sent,bytes_recv,duration,hour_of_day,is_external_dst,dst_port,protocol,upload_download_ratio
3245,0.000182,0.027266,0.000335,17,0,143,0,0.0528
4642,0.006049,0.094616,0.000855,9,1,3306,1,0.5000
1371,0.003355,0.081452,0.009980,9,0,5432,1,0.3222
1634,0.001077,0.122789,0.061673,8,0,53,1,0.0687
706,0.005521,0.126170,0.032925,11,0,3389,0,0.3422


## 3. Huấn luyện Mô hình

- **Logistic Regression (Baseline):** Mô hình phân loại tuyến tính cơ bản.
- **Random Forest (Chính):** Mô hình Decision Tree Ensemble.
- **Isolation Forest:** Mô hình phát hiện bất thường Unsupervised (Không giám sát - chỉ sử dụng nhãn Normal khi train).

In [3]:
from train_models import train_all_models

models = train_all_models(X_train, y_train)

  HUẤN LUYỆN MÔ HÌNH
  Tập train: 4000 bản ghi, 8 đặc trưng

┌─────────────────────────────────────────────────┐
│  MÔ HÌNH 1: LOGISTIC REGRESSION (BASELINE)      │
└─────────────────────────────────────────────────┘
  Cấu hình: solver='lbfgs', max_iter=1000, class_weight='balanced'
  [✓] Huấn luyện hoàn tất

  Hệ số mô hình (|lớn| = quan trọng):
    + duration                  +10.3668  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    + bytes_sent                +9.3532  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    + is_external_dst           +1.9483  ▓▓▓▓▓▓▓▓▓
    - bytes_recv                -1.3444  ▓▓▓▓▓▓
    - protocol                  -0.3676  ▓
    - hour_of_day               -0.0502  
    + upload_download_ratio     +0.0057  
    - dst_port                  -0.0001  

┌─────────────────────────────────────────────────┐
│  MÔ HÌNH 2: RANDOM FOREST (MÔ HÌNH CHÍNH)       │
└─────────────────────────────────────────────────┘
  Cấu hình: n_estimators=100, max_depth=10, 

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.


  [✓] Huấn luyện hoàn tất

  Feature Importance:
    #1 duration                  0.3262  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    #2 bytes_sent                0.3169  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    #3 upload_download_ratio     0.1844  ▓▓▓▓▓▓▓▓▓
    #4 is_external_dst           0.0818  ▓▓▓▓
    #5 hour_of_day               0.0598  ▓▓
    #6 dst_port                  0.0227  ▓
    #7 bytes_recv                0.0075  
    #8 protocol                  0.0007  

┌─────────────────────────────────────────────────┐
│  MÔ HÌNH 3: ISOLATION FOREST (KHÔNG GIÁM SÁT)   │
└─────────────────────────────────────────────────┘
  Cấu hình: n_estimators=100, contamination=0.15
  Chỉ huấn luyện trên dữ liệu Normal (label=0)
  Tập train (chỉ Normal): 3400 bản ghi


  [✓] Huấn luyện hoàn tất

── Lưu mô hình ──
  [✓] Lưu logistic_regression → /home/adc300/src/ptit/atbmtt/outputs/logistic_regression.pkl
  [✓] Lưu random_forest → /home/adc300/src/ptit/atbmtt/outputs/random_forest.pkl
  [✓] Lưu isolation_forest → /home/adc300/src/ptit/atbmtt/outputs/isolation_forest.pkl

  [✓] HUẤN LUYỆN HOÀN TẤT — 3 mô hình


[Parallel(n_jobs=16)]: Done   2 out of  16 | elapsed:    0.1s remaining:    0.4s
[Parallel(n_jobs=16)]: Done  16 out of  16 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    0.0s finished


## 4. Đánh giá Kết quả (Evaluation)

Sử dụng các chỉ số: **Accuracy**, **Precision**, **Recall**, **F1-Score**, **FPR** và biểu đồ **Confusion Matrix**.

In [4]:
from evaluate import evaluate_all
import matplotlib.pyplot as plt
%matplotlib inline

results = evaluate_all(models, X_test, y_test, feature_names=list(X_test.columns))

  ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST
  Tập test: 1000 bản ghi (Normal: 850, Leakage: 150)

  ── Logistic Regression ──
    Accuracy  : 99.5%
    Precision : 98.0%
    Recall    : 98.7%
    F1-Score  : 98.3%
    FPR       : 0.4%
    (TP=148, TN=847, FP=3, FN=2)

  ── Random Forest ──
    Accuracy  : 99.9%
    Precision : 100.0%
    Recall    : 99.3%
    F1-Score  : 99.7%
    FPR       : 0.0%
    (TP=149, TN=850, FP=0, FN=1)

  ── Isolation Forest ──
    Accuracy  : 86.1%
    Precision : 51.9%
    Recall    : 100.0%
    F1-Score  : 68.3%
    FPR       : 16.4%
    (TP=150, TN=711, FP=139, FN=0)

  BẢNG TỔNG HỢP KẾT QUẢ
            Mô hình Accuracy Precision Recall F1-Score   FPR
Logistic Regression    99.5%     98.0%  98.7%    98.3%  0.4%
      Random Forest    99.9%    100.0%  99.3%    99.7%  0.0%
   Isolation Forest    86.1%     51.9% 100.0%    68.3% 16.4%

── Tạo biểu đồ ──


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    0.0s finished


  [✓] Lưu confusion matrices → /home/adc300/src/ptit/atbmtt/outputs/confusion_matrices.png
  [✓] Lưu metrics comparison → /home/adc300/src/ptit/atbmtt/outputs/metrics_comparison.png


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.0s finished


  [✓] Lưu ROC curves → /home/adc300/src/ptit/atbmtt/outputs/roc_curves.png
  [✓] Lưu feature importance → /home/adc300/src/ptit/atbmtt/outputs/feature_importance.png

  [✓] ĐÁNH GIÁ HOÀN TẤT — Kết quả tại outputs/


## 5. Kết luận \& Trực quan hóa

Các biểu đồ kết quả trực quan được xuất ra thư mục `outputs/`:

### So sánh Metrics
![](outputs/metrics_comparison.png)

### Confusion Matrices
![](outputs/confusion_matrices.png)

### ROC Curves
![](outputs/roc_curves.png)

### Feature Importance (Random Forest)
![](outputs/feature_importance.png)

- **Random Forest** cho kết quả tốt nhất khi có dữ liệu gán nhãn, nhận thức rõ nhất các đặc trưng `bytes_sent` và `upload_download_ratio`.
- Mô hình phát hiện rò rỉ dữ liệu nhận biệt rõ được hành vi truyền dữ liệu trái phép thông qua các đặc trưng ngữ cảnh (cổng giao tiếp, giờ giấc, tỉ lệ truy xuất ngoài).